---
# Curso: Data Mining - Semana 3

Docente: Soledad Espezúa (s.espezua@up.edu.pe)

Integrantes del grupo: Fernando Torres, Romy Tipacti, Arturo Alvarez

---

# <font color=blue>Juntando las dos fuentes de nuestro proyecto</font>

El MEF sabe cuánto presupuesto tiene cada proyecto de inversión y cuánto se ha gastado, pero esa cifra sola no dice si la obra realmente está avanzando. Por otro lado, el historial de procesos de selección sí registra el avance físico mes a mes, pero como una tabla completamente aparte, sin relación con lo financiero. Sin cruzar las dos cosas es difícil responder algo tan simple como: ¿este proyecto ya gastó el 80% de su presupuesto pero la obra recién va en 20%?

Esa es la pregunta que nos interesa, y a quien más le sirve responderla es a alguien que supervisa estos proyectos —un equipo de seguimiento de inversiones, una contraloría, o el propio equipo del proyecto— y necesita detectar a tiempo los casos donde se está gastando plata sin que la obra avance al mismo ritmo.

Para eso, en este cuaderno revisamos las dos fuentes, entendemos qué representa cada fila, les ponemos nombres de columna que se entiendan, las cruzamos sin perder de vista qué cruza y qué no, y sacamos unas primeras lecturas con gráficos.


# <font color=green>Parte 1. Cargar las fuentes</font>

Primero traemos las dos tablas a memoria.

* `Seguimiento_PI.csv` (MEF): cuánto se presupuestó y se gastó en cada proyecto, por año.
* `PROCESO_SELECCION.csv` (MEF): el historial mes a mes de cómo va avanzando cada obra.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

RUTA_SEGUIMIENTO = "Seguimiento_PI.csv"
RUTA_PROCESO = "PROCESO_SELECCION.csv"

# SECTOR y PLIEGO los leemos como texto porque, como vamos a ver más abajo,
# traen un espacio en blanco en vez de venir vacíos, y eso confunde a pandas.
seguimiento = pd.read_csv(RUTA_SEGUIMIENTO, dtype={"SECTOR": str, "PLIEGO": str})

# el archivo de Proceso_Selección trae un caracter raro al inicio de la
# cabecera (BOM); con utf-8-sig se lee bien. VAL_META_CAPAC y
# DES_OBSERVACIONES también las forzamos a texto porque mezclan tipos.
proceso = pd.read_csv(
    RUTA_PROCESO, encoding="utf-8-sig",
    dtype={"VAL_META_CAPAC": str, "DES_OBSERVACIONES": str},
)

print("Fuente 1: Seguimiento_PI  ->", seguimiento.shape)
print("Fuente 2: Proceso_Selección ->", proceso.shape)

Fuente 1: Seguimiento_PI  -> (697040, 35)
Fuente 2: Proceso_Selección -> (5673189, 14)


### <font color=279CF5>1.1 Diccionario de datos</font>

Antes de seguir, hay algo que conviene arreglar. Nombres como `ANO_EJE`, `SEC_EJEC` o `DES_TIPO_COMPONENTE` son los que trae el archivo original, y sabemos qué significan porque ya habíamos armado un diccionario de datos (`diccionario_de_datos.xlsx`) documentando cada columna de las dos fuentes. Pero para trabajar más cómodos, y para que cualquiera que abra este cuaderno entienda una columna sin tener que ir a buscar el diccionario, les ponemos nombres más claros.

De paso aprovechamos para arreglar algo: `PRODUCTO_PROYECTO` (en Seguimiento_PI) y `CODIGO_UNICO` (en Proceso_Selección) son la misma cosa —el código del proyecto— solo que cada archivo le puso un nombre distinto. Si les damos el mismo nombre nuevo (`codigo_proyecto`) en las dos tablas, el cruce de más adelante queda más simple.

Usamos la columna `variable` del diccionario, le agregamos el nombre nuevo, y guardamos esa tabla aparte también.


In [2]:
# ya teníamos armado un diccionario con las 49 columnas de ambas fuentes,
# así que lo cargamos en vez de escribirlo de nuevo
diccionario = pd.read_excel("diccionario_de_datos.xlsx")

# nombre nuevo para cada columna. CODIGO_UNICO y PRODUCTO_PROYECTO quedan
# con el mismo nombre a propósito: en el fondo son el mismo dato
nombres_claros = {
    # --- Seguimiento_PI ---
    "ANO_EJE": "anio_ejecucion",
    "NIVEL_GOBIERNO": "nivel_gobierno_cod",
    "NIVEL_GOBIERNO_NOMBRE": "nivel_gobierno",
    "SECTOR": "sector_cod",
    "SECTOR_NOMBRE": "sector",
    "PLIEGO": "pliego_cod",
    "PLIEGO_NOMBRE": "pliego",
    "SEC_EJEC": "seccion_ejecutora_cod",
    "EJECUTORA": "entidad_ejecutora_cod",
    "EJECUTORA_NOMBRE": "entidad_ejecutora",
    "DEPARTAMENTO_EJECUTORA": "departamento_ejecutora_cod",
    "DEPARTAMENTO_EJECUTORA_NOMBRE": "departamento_ejecutora",
    "PROVINCIA_EJECUTORA": "provincia_ejecutora_cod",
    "PROVINCIA_EJECUTORA_NOMBRE": "provincia_ejecutora",
    "DISTRITO_EJECUTORA": "distrito_ejecutora_cod",
    "DISTRITO_EJECUTORA_NOMBRE": "distrito_ejecutora",
    "TIPO_ACT_PROY": "tipo_actividad_cod",
    "TIPO_ACT_PROY_NOMBRE": "tipo_actividad",
    "PRODUCTO_PROYECTO": "codigo_proyecto",
    "PRODUCTO_PROYECTO_NOMBRE": "nombre_proyecto",
    "FUNCION": "funcion_cod",
    "FUNCION_NOMBRE": "funcion",
    "DEPARTAMENTO_META": "departamento_meta_cod",
    "DEPARTAMENTO_META_NOMBRE": "departamento_meta",
    "FUENTE_FINANCIAMIENTO": "fuente_financiamiento_cod",
    "FUENTE_FINANCIAMIENTO_NOMBRE": "fuente_financiamiento",
    "RUBRO": "rubro_cod",
    "RUBRO_NOMBRE": "rubro",
    "COSTO_ACTUAL": "costo_actual",
    "MONTO_EJECUCION_HASTA_HACE_2_ANOS": "monto_ejecutado_acumulado_previo",
    "MONTO_EJECUCION_ANO_ANTERIOR": "monto_ejecutado_anio_anterior",
    "MONTO_PIA": "presupuesto_inicial",
    "MONTO_PIM": "presupuesto_modificado",
    "MONTO_DEVENGADO_ANO_EJE": "monto_devengado_anio_actual",
    "MONTO_EJECUCION_TOTAL": "monto_ejecutado_total",
    # --- Proceso_Selección ---
    "CODIGO_UNICO": "codigo_proyecto",
    "DES_PRODUCTO": "descripcion_producto",
    "DES_ACCION": "descripcion_accion",
    "DES_TIPO_COMPONENTE": "tipo_componente",
    "DES_UM_PRODU": "unidad_medida_producto",
    "VAL_META_PRODU": "meta_producto_valor",
    "DES_UM_CAPAC": "unidad_medida_capacidad",
    "VAL_META_CAPAC": "meta_capacidad_valor",
    "COSTO_INVERSION": "costo_inversion",
    "PERIODO": "periodo",
    "VALORIZ_ACUM": "valorizacion_acumulada",
    "AVANCE": "avance_fisico_pct",
    "DES_OBSERVACIONES": "observaciones",
    "DES_ETAPA": "etapa_proyecto",
}

diccionario["nombre_nuevo"] = diccionario["variable"].map(nombres_claros)

diccionario_mejorado = diccionario.rename(columns={"variable": "variable_original"})[
    ["variable_original", "nombre_nuevo", "fuente", "descripcion", "tipo_general",
     "escala", "rol_o_subtipo", "observaciones"]
]
diccionario_mejorado.to_csv("diccionario_nombres_claros.csv", index=False, encoding="utf-8-sig")

print("variables sin nombre nuevo asignado:", diccionario_mejorado["nombre_nuevo"].isna().sum(), "de", len(diccionario_mejorado))
diccionario_mejorado[["variable_original", "nombre_nuevo", "fuente"]].head(10)

variables sin nombre nuevo asignado: 0 de 49


,variable_original,nombre_nuevo,fuente
0,ANO_EJE,anio_ejecucion,Seguimiento_PI
1,NIVEL_GOBIERNO,nivel_gobierno_cod,Seguimiento_PI
2,NIVEL_GOBIERNO_NOMBRE,nivel_gobierno,Seguimiento_PI
3,SECTOR,sector_cod,Seguimiento_PI
4,SECTOR_NOMBRE,sector,Seguimiento_PI
5,PLIEGO,pliego_cod,Seguimiento_PI
6,PLIEGO_NOMBRE,pliego,Seguimiento_PI
7,SEC_EJEC,seccion_ejecutora_cod,Seguimiento_PI
8,EJECUTORA,entidad_ejecutora_cod,Seguimiento_PI
9,EJECUTORA_NOMBRE,entidad_ejecutora,Seguimiento_PI


In [3]:
# aplicamos el cambio de nombre en las dos tablas
seguimiento = seguimiento.rename(columns=nombres_claros)
proceso = proceso.rename(columns=nombres_claros)

print("¿'codigo_proyecto' quedó en las dos tablas?",
      "codigo_proyecto" in seguimiento.columns and "codigo_proyecto" in proceso.columns)

¿'codigo_proyecto' quedó en las dos tablas? True


De aquí para adelante usamos solo los nombres nuevos. Si en algún momento hace falta el nombre original de una columna, `diccionario_nombres_claros.csv` queda como referencia.

### <font color=279CF5>1.2 ¿Qué tan completos y limpios están estos datos?</font>

Antes de integrar nada conviene mirar con calma qué tan sanas están las dos fuentes: si faltan datos, si hay filas repetidas, si las categorías están escritas siempre igual, y si el formato de alguna columna trae algo raro.


In [4]:
print("seguimiento: tipos de dato por columna")
seguimiento.dtypes

anio_ejecucion                        int64
nivel_gobierno_cod                   object
nivel_gobierno                       object
sector_cod                           object
sector                               object
pliego_cod                           object
pliego                               object
seccion_ejecutora_cod                 int64
entidad_ejecutora_cod                 int64
entidad_ejecutora                    object
departamento_ejecutora_cod            int64
departamento_ejecutora               object
provincia_ejecutora_cod               int64
provincia_ejecutora                  object
distrito_ejecutora_cod                int64
distrito_ejecutora                   object
tipo_actividad_cod                    int64
tipo_actividad                       object
codigo_proyecto                       int64
nombre_proyecto                      object
funcion_cod                           int64
funcion                              object
departamento_meta_cod           

In [5]:
print("proceso: tipos de dato por columna")
proceso.dtypes

codigo_proyecto              int64
descripcion_producto        object
descripcion_accion          object
tipo_componente             object
unidad_medida_producto      object
meta_producto_valor        float64
unidad_medida_capacidad     object
meta_capacidad_valor        object
costo_inversion            float64
periodo                     object
valorizacion_acumulada     float64
avance_fisico_pct          float64
observaciones               object
etapa_proyecto              object
dtype: object

**¿Faltan datos?**

En `seguimiento` no aparece ningún valor vacío a simple vista (ya vamos a ver por qué esto es engañoso). En `proceso` sí hay bastante: `avance_fisico_pct` falta en casi el 90% de las filas, y varias columnas de descripción tienen huecos también.


In [6]:
print("valores vacíos en seguimiento:", seguimiento.isna().sum().sum())

faltantes_proceso = pd.DataFrame({
    "cuantos_faltan": proceso.isna().sum(),
    "porcentaje": (proceso.isna().mean() * 100).round(2),
})
faltantes_proceso

valores vacíos en seguimiento: 0


,cuantos_faltan,porcentaje
codigo_proyecto,0,0.00
descripcion_producto,96074,1.69
descripcion_accion,585703,10.33
tipo_componente,490675,8.65
unidad_medida_producto,1464212,25.81
meta_producto_valor,1177544,20.76
unidad_medida_capacidad,2791006,49.20
meta_capacidad_valor,1450097,25.56
costo_inversion,510,0.01
periodo,1418376,25.01


**¿Categorías escritas siempre igual?**

`etapa_proyecto` debería tener 7 valores posibles nada más, pero mira cómo están escritos:


In [7]:
proceso["etapa_proyecto"].value_counts()

etapa_proyecto
CONSISTENCIA              1193602
Ejecución física (C)      1158941
Expediente técnico (B)    1001091
FYE                        981359
EJECUCION                  648548
CRONOGRAMA                 375734
CONTRACTUAL                312999
Name: count, dtype: int64

Hay dos formas de escribir lo mismo: algunas en mayúsculas sin tilde (`EJECUCION`, `CONTRACTUAL`) y otras con mayúscula/minúscula mezclada, tilde y hasta un código entre paréntesis (`Ejecución física (C)`). Si no unificamos esto antes de contar nada, cualquier conteo por etapa va a separar cosas que en realidad son la misma categoría.

**¿Y el "sin faltantes" de seguimiento?**

Dijimos que `seguimiento.isna()` no marca nada, pero eso no quiere decir que esté perfecto:


In [8]:
print("espacios en blanco en sector_cod:", (seguimiento["sector_cod"].str.strip() == "").sum())
print("espacios en blanco en pliego_cod:", (seguimiento["pliego_cod"].str.strip() == "").sum())

seguimiento.loc[seguimiento["sector_cod"].str.strip() == "", "nivel_gobierno"].value_counts()

espacios en blanco en sector_cod: 616784
espacios en blanco en pliego_cod: 616784


nivel_gobierno
GOBIERNOS LOCALES    616784
Name: count, dtype: int64

En vez de venir vacías, esas celdas traen un espacio en blanco — por eso `isna()` no las detecta. Y no es un error: coincide justo con las filas de "GOBIERNOS LOCALES", que de verdad no tienen un código de sector nacional asignado. Es un vacío que tiene sentido, pero como viene disfrazado de espacio en vez de NaN, más abajo lo vamos a dejar como faltante explícito.

**¿Y filas duplicadas?**


In [9]:
print("filas repetidas de principio a fin en proceso:", proceso.duplicated().sum())

filas repetidas de principio a fin en proceso: 113068


Sí hay: 113,068 filas que son exactamente iguales de principio a fin, no una repetición del historial mensual (eso sí es normal, lo vemos en la Parte 4), sino la misma fila cargada dos veces. Esas las sacamos.

Entonces, resumiendo lo que encontramos: **sí hay valores faltantes** (sobre todo en proceso), **sí hay duplicados** (113,068 filas exactas en proceso), **sí hay categorías escritas de forma inconsistente** (`etapa_proyecto`) y **sí hay un problema de formato** (el espacio en blanco en `sector_cod`/`pliego_cod` que se hace pasar por dato completo). Con esto ya sabemos qué arreglar antes de integrar.


### <font color=279CF5>1.3 Los arreglos mínimos antes de seguir</font>

No hace falta una limpieza exhaustiva, pero sí corregir lo que acabamos de encontrar y que nos va a estorbar más adelante.


In [10]:
# sacamos las 113,068 filas duplicadas de verdad (guardamos el total antes,
# nos sirve más adelante para la tabla de validación de claves)
n_proceso_bruto = len(proceso)
n_dup_exactos_proceso = proceso.duplicated().sum()
proceso = proceso.drop_duplicates().copy()

# el espacio en blanco de sector_cod/pliego_cod pasa a ser un NaN de verdad
seguimiento["sector_cod"] = seguimiento["sector_cod"].replace(r"^\s*$", np.nan, regex=True)
seguimiento["pliego_cod"] = seguimiento["pliego_cod"].replace(r"^\s*$", np.nan, regex=True)

# periodo viene como texto "2024-05"; lo pasamos a fecha para poder ordenar
proceso["periodo_fecha"] = pd.to_datetime(proceso["periodo"], format="%Y-%m", errors="coerce")
hoy = pd.Timestamp("2026-08-19")

# avance_fisico_pct debería ser un porcentaje (0-100), costo_inversion no
# debería ser negativo, y periodo no debería caer en el futuro. Cuando pasa,
# lo dejamos como dato faltante en vez de inventar un valor
fuera_avance = (proceso["avance_fisico_pct"] < 0) | (proceso["avance_fisico_pct"] > 100)
fuera_costo = proceso["costo_inversion"] < 0
fuera_periodo = proceso["periodo_fecha"] > hoy

proceso.loc[fuera_avance, "avance_fisico_pct"] = np.nan
proceso.loc[fuera_costo, "costo_inversion"] = np.nan
proceso.loc[fuera_periodo, "periodo_fecha"] = pd.NaT

# unificamos etapa_proyecto a una sola forma de escritura
proceso["etapa_proyecto"] = (
    proceso["etapa_proyecto"].astype(str).str.upper()
    .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii")
    .str.strip()
)

print("proceso sin los duplicados exactos:", proceso.shape)
print("avance_fisico_pct fuera de 0-100, ahora NaN:", fuera_avance.sum())
print("costo_inversion negativo, ahora NaN:", fuera_costo.sum())
print("periodo en el futuro, ahora NaT:", fuera_periodo.sum())
print("\netapa_proyecto ya unificada:")
proceso["etapa_proyecto"].value_counts()

proceso sin los duplicados exactos: (5560121, 15)
avance_fisico_pct fuera de 0-100, ahora NaN: 6090
costo_inversion negativo, ahora NaN: 1027
periodo en el futuro, ahora NaT: 150181

etapa_proyecto ya unificada:


etapa_proyecto
CONSISTENCIA              1192940
EJECUCION FISICA (C)      1049785
EXPEDIENTE TECNICO (B)     999965
FYE                        981121
EJECUCION                  648326
CRONOGRAMA                 375219
CONTRACTUAL                312765
Name: count, dtype: int64

# <font color=green>Parte 2. ¿Qué representa cada fila?</font>

Antes de seguir, hay que responder algo básico: ¿cada fila es una persona, un distrito, una compra, un trámite, una transacción, una institución, un periodo... o qué?

En nuestro caso no es ninguna de esas. Cada fila representa **un proyecto de inversión pública**: una obra o intervención específica, identificada por su código único de inversión (el mismo que usa el Banco de Inversiones / Invierte.pe). No es la institución que ejecuta el proyecto (una misma entidad puede tener muchos proyectos), ni un trámite puntual, ni una transacción de un momento dado: es el proyecto completo, visto a lo largo del tiempo.

El problema es que, tal cual vienen, ninguna de las dos tablas está a nivel de proyecto:

- `seguimiento` repite el proyecto una vez por cada año, pliego y fuente de financiamiento (697,040 filas para 52,800 proyectos distintos).
- `proceso` repite el proyecto una vez por cada mes que fue reportado (5,673,189 filas para 204,209 proyectos distintos).

Por eso no alcanza con un `merge` directo: antes hay que decidir qué fila de cada tabla representa "el proyecto" (eso lo resolvemos en la Parte 4).


### <font color=279CF5>2.1 ¿Con qué variables contamos?</font>

Entre las dos fuentes tenemos 49 columnas (ya documentadas en el diccionario), pero no todas pesan igual para el análisis. Estas son las que realmente vamos a usar:


In [11]:
variables_clave = pd.DataFrame({
    "variable": ["codigo_proyecto", "nombre_proyecto", "nivel_gobierno", "presupuesto_modificado",
                 "monto_ejecutado_total", "costo_inversion", "avance_fisico_pct", "etapa_proyecto"],
    "tipo": ["identificador", "texto", "categórica", "numérica", "numérica", "numérica",
             "numérica (%)", "categórica ordinal"],
    "para_que_sirve": [
        "es la llave que nos deja juntar las dos fuentes",
        "para saber de qué obra estamos hablando",
        "para comparar entre gobierno nacional, regional y local",
        "cuánto presupuesto tiene aprobado el proyecto",
        "cuánto se gastó realmente (el lado financiero, según el MEF)",
        "el costo del proyecto según el expediente técnico",
        "qué tan avanzada está la obra en la realidad (el lado físico)",
        "en qué momento del ciclo del proyecto está (expediente, ejecución, etc.)",
    ],
})
variables_clave

,variable,tipo,para_que_sirve
0,codigo_proyecto,identificador,es la llave que nos deja juntar las dos fuentes
1,nombre_proyecto,texto,para saber de qué obra estamos hablando
2,nivel_gobierno,categórica,"para comparar entre gobierno nacional, regional y local"
3,presupuesto_modificado,numérica,cuánto presupuesto tiene aprobado el proyecto
4,monto_ejecutado_total,numérica,"cuánto se gastó realmente (el lado financiero, según el MEF)"
5,costo_inversion,numérica,el costo del proyecto según el expediente técnico
6,avance_fisico_pct,numérica (%),qué tan avanzada está la obra en la realidad (el lado físico)
7,etapa_proyecto,categórica ordinal,"en qué momento del ciclo del proyecto está (expediente, ejecución, etc.)"


¿Alcanza esto para comparar, agrupar, asociar o detectar patrones? Sí:

- **Comparar**: el avance promedio entre gobierno nacional, regional y local (`nivel_gobierno` vs. `avance_fisico_pct`).
- **Agrupar**: cuántos proyectos hay y cuánto se gastó por nivel de gobierno o por etapa.
- **Asociar**: si gastar más plata (`monto_ejecutado_total`) va de la mano con más avance físico (`avance_fisico_pct`), o no.
- **Detectar patrones raros**: proyectos que ya ejecutaron mucho presupuesto pero siguen con poco avance físico — justo el tipo de caso que le interesa a alguien que supervisa estos proyectos.


# <font color=green>Parte 3. El atributo de unión</font>

En el archivo original cada fuente la llamaba distinto; después de renombrar, es literalmente la misma columna en las dos tablas.


In [12]:
tabla_clave = pd.DataFrame({
    "Fuente": ["Seguimiento_PI.csv (MEF)", "PROCESO_SELECCION.csv (MEF)"],
    "Contenido": ["Ejecución presupuestal por año, pliego y fuente de financiamiento",
                  "Historial mensual de avance físico y contractual"],
    "Llave (nombre original)": ["PRODUCTO_PROYECTO", "CODIGO_UNICO"],
    "Llave (nombre nuevo)": ["codigo_proyecto", "codigo_proyecto"],
})
tabla_clave

,Fuente,Contenido,Llave (nombre original),Llave (nombre nuevo)
0,Seguimiento_PI.csv (MEF),"Ejecución presupuestal por año, pliego y fuente de financiamiento",PRODUCTO_PROYECTO,codigo_proyecto
1,PROCESO_SELECCION.csv (MEF),Historial mensual de avance físico y contractual,CODIGO_UNICO,codigo_proyecto


La clave de integración es **`codigo_proyecto`**, y está en las dos tablas con el mismo nombre — así el `merge` de la Parte 6 se puede escribir con un simple `on="codigo_proyecto"`.


# <font color=green>Parte 4. ¿Qué tanto se repite la clave?</font>

Antes de cruzar, miramos cuántas veces aparece `codigo_proyecto` en cada tabla. Como esperamos una fila por proyecto, cualquier repetición hay que poder explicarla.


In [13]:
rep_seguimiento = seguimiento.groupby("codigo_proyecto").size()
rep_proceso = proceso.groupby("codigo_proyecto").size()

print("codigo_proyecto en seguimiento:")
print("  valores únicos:", seguimiento["codigo_proyecto"].nunique(), "de", len(seguimiento), "filas")
print("  se repite en promedio", round(rep_seguimiento.mean(), 2), "veces, y hasta", rep_seguimiento.max(),
      "veces (código", rep_seguimiento.idxmax(), ")")

print("\ncodigo_proyecto en proceso:")
print("  valores únicos:", proceso["codigo_proyecto"].nunique(), "de", len(proceso), "filas")
print("  se repite en promedio", round(rep_proceso.mean(), 2), "veces, y hasta", rep_proceso.max(),
      "veces (código", rep_proceso.idxmax(), ")")

codigo_proyecto en seguimiento:
  valores únicos: 52800 de 697040 filas
  se repite en promedio 13.2 veces, y hasta 82864 veces (código 2001621 )

codigo_proyecto en proceso:
  valores únicos: 204209 de 5560121 filas
  se repite en promedio 27.23 veces, y hasta 7656 veces (código 2307923 )


¿Es esto un problema? Depende de por qué se repite:

- En `seguimiento`, cada proyecto aparece una vez por año/pliego/fuente de financiamiento, así que la repetición es normal. El caso más extremo (código `2001621`, se repite 82,864 veces) ni siquiera es un proyecto de verdad: es "Estudios de Pre-Inversión", un código genérico que agrupa varios estudios sin obra asociada. Por eso tampoco aparece en `proceso`.
- En `proceso`, cada proyecto aparece una vez por mes reportado, así que también es normal. El proyecto `2307923` es el que lleva más historial: 7,656 filas, casi 20 años de reportes mes a mes.

Lo que sí importa es lo siguiente: si cruzamos las tablas tal como están, cada fila de `seguimiento` se multiplicaría por todo el historial mensual que tenga ese proyecto en `proceso`. Con miles de filas de historial por proyecto, terminaríamos con una tabla gigante y sin sentido. Antes de integrar, hay que reducir `proceso` a **una sola fila por proyecto: la del mes más reciente**.


In [14]:
# separamos los proyectos que sí tienen un periodo que se pudo leer bien
con_periodo = proceso[proceso["periodo_fecha"].notna()].copy()
sin_periodo = proceso[proceso["periodo_fecha"].isna()].copy()

codigos_con_periodo = set(con_periodo["codigo_proyecto"].unique())
codigos_totales = set(proceso["codigo_proyecto"].unique())

# para los que sí tienen periodo, nos quedamos con el más reciente
con_periodo = con_periodo.sort_values("periodo_fecha")
snapshot_reciente = con_periodo.drop_duplicates(subset="codigo_proyecto", keep="last").copy()
snapshot_reciente["periodo_disponible"] = True

# para los que nunca tuvieron un periodo legible, nos quedamos con su última
# fila tal cual, pero marcada: no podemos decir cuál es "la más reciente"
sin_periodo_agg = sin_periodo[~sin_periodo["codigo_proyecto"].isin(codigos_con_periodo)].copy()
sin_periodo_agg = sin_periodo_agg.drop_duplicates(subset="codigo_proyecto", keep="last").copy()
sin_periodo_agg["periodo_disponible"] = False

snapshot_proceso = pd.concat([snapshot_reciente, sin_periodo_agg], ignore_index=True)

print("snapshot final (una fila por proyecto):", snapshot_proceso.shape)
print("¿quedó una sola fila por proyecto?", snapshot_proceso["codigo_proyecto"].is_unique)
print("proyectos con un periodo confiable:", round(snapshot_proceso["periodo_disponible"].mean() * 100, 2), "%")

snapshot final (una fila por proyecto): (204209, 16)
¿quedó una sola fila por proyecto? True
proyectos con un periodo confiable: 93.89 %


Ahora `snapshot_proceso` tiene una fila por cada uno de los 204,209 proyectos que aparecen en `proceso`. Para el 93.89% sabemos que es de un mes real; para el resto (12,478 proyectos) lo marcamos con `periodo_disponible = False`, para no confundirlo con un dato confirmado. Esta es la tabla que vamos a cruzar, no el archivo original.


# <font color=green>Parte 5. Cruzar información</font>

Usamos `how="outer"` para ver los tres casos a la vez: lo que cruza, lo que solo está en `seguimiento` y lo que solo está en `proceso`.


In [15]:
auditoria = pd.merge(
    seguimiento[["codigo_proyecto"]],
    snapshot_proceso[["codigo_proyecto"]],
    on="codigo_proyecto",
    how="outer",
    indicator="origen_cruce",   # agrega una columna que dice de dónde vino cada fila
)

conteo_cruce = auditoria["origen_cruce"].value_counts().reset_index()
conteo_cruce.columns = ["resultado_cruce", "cantidad"]
conteo_cruce

,resultado_cruce,cantidad
0,both,172708
1,left_only,524332
2,right_only,168667


In [16]:
fig = px.bar(
    conteo_cruce, x="resultado_cruce", y="cantidad", text="cantidad",
    color="resultado_cruce",
    color_discrete_map={"both": "#2E7D32", "left_only": "#F9A825", "right_only": "#C62828"},
    title="Resultado de la integración: seguimiento vs. proceso",
)
fig.show()

<Figure Plotly>

En números: **172,708** filas cruzaron bien, **524,332** están solo en `seguimiento` y **168,667** solo en `proceso`.

Ojo con algo: si medimos la cobertura por fila da 24.78%, pero si la medimos por proyecto (¿cuántos proyectos distintos de seguimiento también están en proceso?) sube a 67.3% (35,542 de 52,800). La diferencia se explica porque los proyectos que NO cruzan son, en promedio, justo los que más se repiten en `seguimiento` — como el código genérico que vimos en la Parte 4. Conviene tener esto presente antes de sacar conclusiones con los números agregados.


# <font color=green>Parte 6. La base integrada</font>

Acá usamos `how="left"` en vez de `how="inner"`: queremos conservar todas las filas de `seguimiento` (esa es la fuente que define qué es un registro en el análisis) y sumarle el avance físico cuando exista, no perder filas.


In [17]:
base_integrada = pd.merge(
    seguimiento,
    snapshot_proceso,
    on="codigo_proyecto",
    how="left",
    suffixes=("_seguimiento", "_proceso"),
    indicator="origen_cruce",
)

print("seguimiento original:", seguimiento.shape)
print("base_integrada:", base_integrada.shape)
print("¿se mantuvo el mismo número de filas que seguimiento?", base_integrada.shape[0] == seguimiento.shape[0])

base_integrada["origen_cruce"].value_counts()

seguimiento original: (697040, 35)
base_integrada: (697040, 51)
¿se mantuvo el mismo número de filas que seguimiento? True


origen_cruce
left_only     524332
both          172708
right_only         0
Name: count, dtype: int64

Un ejemplo real de filas que sí cruzaron:


In [18]:
base_integrada.loc[
    base_integrada["origen_cruce"] == "both",
    ["codigo_proyecto", "nombre_proyecto", "avance_fisico_pct", "costo_inversion",
     "periodo_fecha", "periodo_disponible", "etapa_proyecto", "origen_cruce"]
].head()

,codigo_proyecto,nombre_proyecto,avance_fisico_pct,costo_inversion,periodo_fecha,periodo_disponible,etapa_proyecto,origen_cruce
0,2015918,PROYECTO VIAL TACNA LA PAZ,100.00,168602080.0,2019-11-01,True,EJECUCION,both
1,2022551,MEJORAMIENTO DE CARRETERA VECINAL 533-EMPALME-534-LA ISLILLA,81.28,68867310.0,2023-07-01,True,EJECUCION,both
2,2045193,FORTALECIMIENTO DE LAS CAPACIDADES PARA EL ORDENAMIENTO TERRITORIAL DEL DEPARTAMENTO DE MOQUEGUA,NaN,13236185.0,2023-12-01,True,CRONOGRAMA,both
3,2031741,"PROGRAMA DE GESTION INTEGRAL DE LA CUENCA DE ABANCAY, APURIMAC",83.73,5717098.5,2026-07-01,True,EJECUCION,both
4,2040186,MEJORAMIENTO DE RIEGO Y GENERACION HIDROENERGETICO DEL ALTO PIURA,NaN,153617100.0,2025-01-01,True,CONSISTENCIA,both


### <font color=279CF5>6.1 Tabla de validación de claves</font>

Un resumen de lo visto en la Parte 4, ya con el snapshot incluido.


In [19]:
tabla_validacion = pd.DataFrame([
    ["seguimiento (bruto)", "codigo_proyecto", seguimiento.shape[0],
     seguimiento["codigo_proyecto"].nunique(), round(rep_seguimiento.mean(), 2),
     rep_seguimiento.max(), seguimiento.duplicated().sum(),
     "se repite por diseño (año/pliego/fuente); el caso extremo es un código genérico, no un proyecto real"],
    ["proceso (bruto)", "codigo_proyecto", n_proceso_bruto,
     proceso["codigo_proyecto"].nunique(), round(rep_proceso.mean(), 2),
     rep_proceso.max(), n_dup_exactos_proceso,
     "se repite por diseño (historial mensual); además tenía duplicados exactos que ya sacamos"],
    ["snapshot_proceso (ya reducido)", "codigo_proyecto", snapshot_proceso.shape[0],
     snapshot_proceso.shape[0], 1.0, 1, 0,
     "clave única, como se necesita para cruzar"],
], columns=["fuente", "clave", "n_filas", "n_valores_unicos", "repeticion_promedio",
            "repeticion_maxima", "duplicados_exactos", "interpretacion"])

tabla_validacion

,fuente,clave,n_filas,n_valores_unicos,repeticion_promedio,repeticion_maxima,duplicados_exactos,interpretacion
0,seguimiento (bruto),codigo_proyecto,697040,52800,13.20,82864,0,"se repite por diseño (año/pliego/fuente); el caso extremo es un código genérico, no un proyecto real"
1,proceso (bruto),codigo_proyecto,5673189,204209,27.23,7656,113068,"se repite por diseño (historial mensual); además tenía 113,068 duplicados exactos que ya sacamos"
2,snapshot_proceso (ya reducido),codigo_proyecto,204209,204209,1.00,1,0,"clave única, como se necesita para cruzar"


# <font color=green>Parte 7. Gráficos</font>

El del cruce (Parte 5) ya cuenta como el primero, así que van tres más — uno por cada tipo de variable que nos interesa mirar.


### <font color=279CF5>7.1 Histograma: distribución del avance físico</font>

Este ya sale de las dos fuentes juntas: nos quedamos con las filas de `base_integrada` que sí cruzaron y que además tienen dato de avance.


In [20]:
base_avance = base_integrada[(base_integrada["origen_cruce"] == "both") & base_integrada["avance_fisico_pct"].notna()]
print("filas con cruce y avance físico conocido:", len(base_avance))

fig = px.histogram(
    base_avance,
    x="avance_fisico_pct",
    nbins=20,
    range_x=[0, 100],
    title="Distribución del avance físico — proyectos con cruce entre seguimiento y proceso",
)
fig.show()

filas con cruce y avance físico conocido: 15623


<Figure Plotly>

El salto en el último tramo (95%-100%) es todavía más marcado que antes de cruzar: casi la mitad de estos proyectos (44%) tiene un avance entre 95% y 100%. Tiene sentido — son justo los proyectos que además tienen registro financiero, así que en promedio llevan más tiempo de ejecución que el resto.


### <font color=279CF5>7.2 Boxplot: avance físico por nivel de gobierno</font>

Usamos `base_avance` (ya la teníamos filtrada del gráfico anterior) y sacamos hasta 1,500 proyectos por grupo para que no se sature.


In [21]:
# sacamos hasta 1500 filas de cada grupo por separado y las juntamos. Lo
# hacemos así (con un for en vez de groupby().apply()) para que funcione
# igual sin importar la versión de pandas que tengamos instalada
partes = []
for nivel, grupo in base_avance.groupby("nivel_gobierno"):
    partes.append(grupo[["nivel_gobierno", "avance_fisico_pct"]].sample(min(1500, len(grupo)), random_state=42))
muestra_box = pd.concat(partes, ignore_index=True)

fig = px.box(
    muestra_box, x="nivel_gobierno", y="avance_fisico_pct", color="nivel_gobierno",
    points="outliers",
    title="Avance físico según nivel de gobierno (proyectos con cruce)",
)
fig.show()

<Figure Plotly>

Los tres niveles de gobierno tienen un avance promedio parecido: nacional 79.9%, local 79.2%, regional 76.8%. Ninguno se atrasa mucho más que los otros, al menos en promedio — la diferencia real está más en cuánta plata mueve cada uno (lo vemos en el siguiente gráfico).


### <font color=279CF5>7.3 Dispersión: Monto ejecutado vs. avance físico</font>

De nuevo con una muestra (proyectos con cruce, monto y costo de inversión válidos).


In [22]:
elegibles = base_integrada[
    (base_integrada["origen_cruce"] == "both")
    & (base_integrada["monto_ejecutado_total"] > 0)
    & (base_integrada["costo_inversion"] > 0)
    & base_integrada["avance_fisico_pct"].notna()
]
muestra_scatter = elegibles.sample(min(3000, len(elegibles)), random_state=42)

fig = px.scatter(
    muestra_scatter, x="monto_ejecutado_total", y="avance_fisico_pct",
    color="nivel_gobierno", size="costo_inversion", log_x=True,
    hover_data=["nombre_proyecto", "etapa_proyecto"],
    title="Monto ejecutado vs. avance físico (muestra de proyectos con cruce)",
)
fig.show()

<Figure Plotly>

No se ve una relación clara entre cuánta plata se ejecutó y qué tan avanzada está la obra: el gobierno nacional gasta en promedio 10 veces más por proyecto que el gobierno local (S/ 4.2 millones vs. S/ 0.4 millones), pero el avance físico promedio es casi el mismo. Gastar más no es garantía de más avance — probablemente depende más de en qué etapa está cada proyecto.


# <font color=green>Parte 8. Hallazgos</font>

Cuatro cosas que nos parecieron interesantes al revisar todo esto.


In [23]:
hallazgos = pd.DataFrame({
    "grafico_o_tabla": [
        "Resultado de la integración",
        "Faltantes en columnas incorporadas",
        "Distribución del avance físico",
        "Repetición extrema de una clave",
    ],
    "observacion": [
        "24.8% de match por fila, pero 67.3% por proyecto "
        "(35,542 de 52,800 proyectos de seguimiento también están en proceso).",
        "De los 172,708 registros que sí cruzan, 157,085 (91%) tienen avance_fisico_pct vacío.",
        "El avance físico solo aparece cuando etapa_proyecto = 'EJECUCION'; "
        "en las otras 6 etapas el campo viene vacío.",
        "El código 2001621 ('Estudios de Pre-Inversión') aparece 82,864 veces en "
        "seguimiento (11.9% de las filas) y no existe en proceso.",
    ],
    "posible_hallazgo_o_hipotesis": [
        "Los proyectos que no tienen proceso de selección registrado son, en promedio, "
        "los que más se repiten en seguimiento (posiblemente proyectos antiguos o de bajo monto).",
        "El avance físico solo se registra una vez que el proyecto entra en ejecución de "
        "obra; no es un error, es parte normal del ciclo del proyecto.",
        "Confirma lo anterior: avance_fisico_pct es un dato propio de la etapa EJECUCION, "
        "no algo que se mida en todo el ciclo del proyecto.",
        "No es un proyecto real sino un código presupuestal genérico; conviene tratarlo "
        "aparte en cualquier análisis que asuma un proyecto único por código.",
    ],
    "precaucion": [
        "Habría que comparar monto promedio y antigüedad entre proyectos con y sin cruce.",
        "No hay que leer las otras etapas como '0% de avance': el dato viene vacío por "
        "diseño, no porque el proyecto esté en cero.",
        "El avance promedio por nivel de gobierno describe solo al ~9% de proyectos que "
        "ya llegaron a ejecución, no a todo el portafolio.",
        "Si no se separa este código, se exagera la proporción real de 'proyectos sin cruce'.",
    ],
})

hallazgos

,grafico_o_tabla,observacion,posible_hallazgo_o_hipotesis,precaucion
0,Resultado de la integración,"24.8% de match por fila, pero 67.3% por proyecto (35,542 de 52,800 proyectos de seguimiento también están en proceso).","Los proyectos que no tienen proceso de selección registrado son, en promedio, los que más se repiten en seguimiento (posiblemente proyectos antiguos o de bajo monto).",Habría que comparar monto promedio y antigüedad entre proyectos con y sin cruce antes de afirmar esto con seguridad.
1,Faltantes en columnas incorporadas,"De los 172,708 registros que sí cruzan, 157,085 (91%) tienen avance_fisico_pct vacío.","El avance físico solo se registra una vez que el proyecto entra en ejecución de obra; no es un error, es parte normal del ciclo del proyecto (perfil -> expediente -> ejecución).","No hay que leer las otras etapas como '0% de avance': el dato viene vacío por diseño, no porque el proyecto esté en cero."
2,Distribución del avance físico,El avance físico solo aparece cuando etapa_proyecto = 'EJECUCION'; en las otras 6 etapas el campo viene vacío.,"Confirma lo anterior: avance_fisico_pct es un dato propio de la etapa EJECUCION, no algo que se mida en todo el ciclo del proyecto.","El avance promedio por nivel de gobierno describe solo al ~9% de proyectos que ya llegaron a ejecución, no a todo el portafolio."
3,Repetición extrema de una clave,"El código 2001621 ('Estudios de Pre-Inversión') aparece 82,864 veces en seguimiento (11.9% de las filas) y no existe en proceso.",No es un proyecto real sino un código presupuestal genérico; conviene tratarlo aparte en cualquier análisis que asuma que cada código es un proyecto único.,"Si no se separa este código, se exagera la proporción real de 'proyectos sin cruce'."


# <font color=green>Parte 9. Guardar todo para la próxima semana</font>


In [24]:
base_integrada.to_csv("base_integrada_actividad_final.csv", index=False, encoding="utf-8-sig")
print("archivo generado: base_integrada_actividad_final.csv", base_integrada.shape)

archivo generado: base_integrada_actividad_final.csv (697040, 51)


## En resumen

Nos quedamos con el diccionario de nombres claros (`diccionario_nombres_claros.csv`), la base integrada (`base_integrada_actividad_final.csv`), la tabla de validación de claves de la 6.1, los cuatro gráficos con su lectura, y los cuatro hallazgos de la Parte 8. Con esto ya tenemos una primera foto completa del cruce entre lo que se presupuesta y lo que realmente avanza en obra.

Lo que nos gustaría construir con esto más adelante no es solo describir promedios, sino algo más útil: una forma de detectar automáticamente los proyectos donde el gasto va muy adelantado respecto al avance físico real — justo el tipo de caso que le interesa a quien supervisa estos proyectos.
